# FinanceBench RAG Pipeline (Colab)

This notebook runs the existing project modules in Google Colab.

It supports:
- Fresh run in Colab (rebuild vectorstore)
- Reuse existing `vectorstore/` from Google Drive
- Smoke run on a small subset before full execution

## 1) Colab Setup

Install dependencies and mount Google Drive.

In [ ]:
%pip -q install -U pip
%pip -q install datasets langchain langchain-community langchain-core langchain-text-splitters
%pip -q install faiss-cpu sentence-transformers pypdf requests cryptography ragas python-dotenv openai pandas tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2) Get Project Code in Colab

Option A: clone from GitHub.

Option B: use a copy already in Google Drive.

In [ ]:
import os
from pathlib import Path

# Set one of these:
GITHUB_REPO_HTTPS = "https://github.com/:Maorb23/rag_project.git"
USE_DRIVE_COPY = False
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/rag_project"

if USE_DRIVE_COPY:
    project_root = Path(DRIVE_PROJECT_PATH)
    assert project_root.exists(), f"Drive project path not found: {project_root}"
else:
    project_root = Path("/content/rag_project")
    if not project_root.exists():
        !git clone $GITHUB_REPO_HTTPS /content/rag_project

os.chdir(project_root)
print("Project root:", project_root)
!ls

## 3) Configure Environment Variables

Set your Nebius credentials and models for generation/judging.

Do not commit secrets to GitHub.

In [ ]:
import os

os.environ["FINANCEBENCH_DATASET_ID"] = "PatronusAI/financebench"
os.environ["FINANCEBENCH_DATASET_SPLIT"] = "train"

# Fill these before running model calls.
os.environ["NEBIUS_BASE_URL"] = "https://api.tokenfactory.nebius.com/v1"
os.environ["NEBIUS_API_KEY"] = ""
os.environ["GENERATION_MODEL"] = "meta-llama/Llama-3.3-70B-Instruct"
os.environ["JUDGE_MODEL"] = "deepseek-ai/DeepSeek-V3.2"
os.environ["RAGAS_MODEL"] = "deepseek-ai/DeepSeek-V3.2"

os.environ["EMBEDDING_MODEL"] = "BAAI/bge-small-en-v1.5"
os.environ["CHUNK_SIZE"] = "1000"
os.environ["CHUNK_OVERLAP"] = "150"
os.environ["RETRIEVAL_DEFAULT_K"] = "4"
os.environ["RETRIEVAL_HIT_K_VALUES"] = "1,3,5"

os.environ["DATA_DIR"] = "data"
os.environ["PDF_DIR"] = "data/pdfs"
os.environ["VECTORSTORE_DIR"] = "vectorstore"
os.environ["RESULTS_DIR"] = "results"

print("Environment variables configured.")

## 4) Optional: Reuse Existing Vectorstore

If you copied your local `vectorstore/` to Google Drive, sync it into the project to avoid rebuilding embeddings.

In [ ]:
from pathlib import Path
import shutil

# Example source in Drive after manual upload:
# /content/drive/MyDrive/rag_project/vectorstore
DRIVE_VECTORSTORE_PATH = "/content/drive/MyDrive/rag_project/vectorstore"

src_vs = Path(DRIVE_VECTORSTORE_PATH)
dst_vs = Path("vectorstore")

if src_vs.exists() and (src_vs / "index.faiss").exists() and (src_vs / "index.pkl").exists():
    dst_vs.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src_vs / "index.faiss", dst_vs / "index.faiss")
    shutil.copy2(src_vs / "index.pkl", dst_vs / "index.pkl")
    print("Copied existing vectorstore from Drive.")
else:
    print("No reusable vectorstore found in Drive path. A new one will be built.")

## 5) Imports and Config

Load project modules and inspect config.

In [ ]:
import os
import sys

sys.path.append(str(project_root))

from src.financebench_rag.config import load_config
from src.financebench_rag.pipeline import execute_full_pipeline
from src.financebench_rag.dataset import prepare_stage1_dataset

config = load_config(env_file=None)
print(config)

## 6) Smoke Run (Optional)

Run only Stage 1 to validate dataset and connectivity quickly.

In [ ]:
filtered_df, mapping_df = prepare_stage1_dataset(
    dataset_id=config.dataset_id,
    split=config.dataset_split,
    output_dir=config.results_dir,
)
print("Filtered rows:", len(filtered_df))
filtered_df[["financebench_id", "question_type", "doc_name"]].head()

## 7) Full Pipeline Run

This executes all stages and writes artifacts to `results/` and `vectorstore/`.

In [ ]:
results = execute_full_pipeline(config)
results["metrics"]

## 8) Copy Artifacts Back to Drive (Optional)

In [ ]:
from pathlib import Path
import shutil

DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/rag_project_colab_outputs")
DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

for folder_name in ["results", "vectorstore"]:
    src = Path(folder_name)
    dst = DRIVE_OUTPUT_ROOT / folder_name
    if src.exists():
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f"Copied {src} -> {dst}")